# Documentation Notebook

Current artifact: `docs/multipole_moment_mmm/ACA_MOMENT_DESIGN.ipynb`.

Legacy Markdown source was promoted and removed.


In [1]:
# Version stamp for embedded results
import json, platform, sys
try:
    import radia
    radia_version = getattr(radia, '__version__', 'unknown')
except Exception as exc:
    radia_version = f'not-importable: {exc}'
version_stamp = {
    'generated_at_utc': '2026-06-25T07:18:58Z',
    'generator_python': '3.12.10',
    'generator_platform': 'Windows-2022Server-10.0.20348-SP0',
    'generator_radia_version': '4.94.0',
    'generator_git_commit': '9bc4b2bb',
    'runtime_python': sys.version.split()[0],
    'runtime_platform': platform.platform(),
    'runtime_radia_version': radia_version,
}
print(json.dumps(version_stamp, indent=2, ensure_ascii=False))


{
  "generated_at_utc": "2026-06-25T07:18:58Z",
  "generator_python": "3.12.10",
  "generator_platform": "Windows-2022Server-10.0.20348-SP0",
  "generator_radia_version": "4.94.0",
  "generator_git_commit": "9bc4b2bb",
  "runtime_python": "3.12.10",
  "runtime_platform": "Windows-2022Server-10.0.20348-SP0",
  "runtime_radia_version": "4.94.0"
}


# Multipole-moment MMM: symbolic moment formulation and scalable solve

**Status (2026-07-02, current):** `rad.Solve(..., method=2)` on a PURE-HEX moment object runs
**HACApK-BiCGSTAB, matvec-only** -- the chi-free geometry coupling K is built once as a
`RadHACApKMomentSystem` H-matrix (O(N log N) matvec + storage) and CACHED CROSS-SOLVE on
`radTApplication` (validity = interaction ptr + hacapk eps/leaf/eta + bit-exact centroid compare), so an
optimization inner loop re-Solving the same geometry reuses K, localL, and diagK across `rad.Solve` calls
(measured 20.5x warm-solve speedup at 1296 DOF; warm re-solve after a mu_r change matches dense LU to
1.9e-14). **Loop-free is abandoned on this route** (Sugahara 2026-06-30): the internal M is field-correct
but loop-polluted -- acceptable for the COARSE / optimization tier; accurate + hysteresis work uses the
loop-free HDiv-VIM (the PRIMARY soft-iron method). NO H-LU, NO deflation (the no-pivot H-LU is wrong AND
slow at compression<1 -- see `memory/collocation_loopfree_abandoned.md`). tet/wedge/mixed method-2 fall
back to the dense moment LU (the moment H-matrix is hex-only). The 3-way solver benchmark (dense LU /
dense-K BiCGSTAB / HACApK-BiCGSTAB; compact cube + elongated bar + voxelized C-yoke) is
`examples/vim/bench_moment_solvers.py`.  The ANALYTIC closed-form moment kernel is the DEFAULT
since the same day (exact + 1.5x faster H-matrix build; Gauss selectable via
`rad.SolverConfig(moment_analytic_kernel=False)`).

*History:* the 2026-06-23 Phase-2 moment H-matrix (with H-LU + two-sided deflation) was REMOVED on
2026-06-28 ("MMMM does not connect to HACApK", dense-only interlude) after the H-LU / loop-free
scalability route was measured dead; the matvec-only H-matrix was REVIVED on 2026-07-02 when giving up
loop-free made the plain HACApK-BiCGSTAB route sound for the coarse tier. The design content below
(moment functionals, block-Jacobi choice, admissibility) is unchanged and still accurate; references to
deflation / H-LU in older sections are historical.

## Naming and scope (2026-06-24)

Do not use the old Yano-centered label for this path.  The production idea is broader than a
particular six-face surface-charge element: it is a **multipole-moment reformulation of MMM**.  The same
principle improves both the classical 3-DOF MMM basis and the 5/6-DOF
surface-charge basis.  The surface-charge case is the most visible one because
six face charges decompose into exactly

    monopole(1) + dipole(3) + residual quadrupole(2).

This decomposition was derived symbolically in Mathematica/Wolfram scripts, and
that derivation is the design anchor: the element is closed by moment conditions
(neutrality, centroid field, centroid field-gradient), not by an empirical
evaluation point.  In contrast, the HDiv Galerkin route gives a symmetric
de-Rham-exact matrix and high-order elements, but its charge-Coulomb Gram
integrals are the expensive part of matrix construction.  The multipole-moment
MMM route keeps the row functionals local and cheap, which is why it is the
production surface-charge path while HDiv-VIM remains the symmetric/high-order
complement.

## What was built

A scalable solver for the multipole-moment MMM linear system so that `rad.Solve(..., method=2)` (HACApK) on pure
6-DOF hex soft iron solves the **moment** system (not EIEM2) with `O(N log N)` storage + matvec. The shipped
local preconditioner is **element-wise block Jacobi**: it is the best production choice among local diagonal
preconditioners because it exactly removes each element's moment constraint block, while scalar Jacobi leaves
the local moment coupling unresolved and larger neighboring super-blocks were empirically worse. Iterations
still grow with the high-mu demag conditioning wall, so a global/hierarchical preconditioner remains future
work. The nonlinear outer loop (Picard / Anderson) wraps this linear solve unchanged.

Benchmark realism policy: linear `mu_r >= 10000` cases are conditioning stress tests, not the main
engineering target. Production claims should be based primarily on nonlinear BH-curve cube/C-yoke runs and
linear reference cases in the ordinary soft-iron range (`mu_r ~ 100` to `5000`, with `mu_r ~ 1000` as the
common lock/reference scale). A preconditioner that only helps the `mu_r >= 10000` stress test stays
experimental unless it also improves the nonlinear or `mu_r ~ 1000` cases.
The durable two-stage smoke record is in `docs/multipole_moment_mmm/MEMORY.md`: method 2 block-Jacobi
and a 3-mode coarse-correction probe were compared against method 0 references.  The probe increased
BiCGSTAB iterations in every small case (`19->28`, `532->568`, `31->36`, `59->69`), so that implementation
and its `SolverConfig` knob were removed.  Future global/hierarchical preconditioning must be redesigned
and revalidated from Stage A, not kept as a dormant option.

Method 1 is no longer a dense comparison path.  Pure 6-DOF hex now applies `A(chi)x = Lx + chi*Kx`
matrix-free through `MomentSystemBlock6x6`, builds only element-wise 6x6 block-Jacobi inverses, and removes
the dense `SystemMatrix` allocation from method-1 BiCGSTAB.  Wedge/pyramid and mixed 5/6-DOF models use the
same storage pattern through `MomentSystemBlockAny`, a padded 6x6 on-demand API whose active block is each
element pair's natural 5x5, 5x6, 6x5, or 6x6 moment block.

TaskManager parallelism is part of the shipped method-2 contract. `RadHACApKBase::BuildHMatrix` and
`SolveMomentHACApK` stand up or reuse an NGSolve `RegionTaskManager`, and the HACApK C callbacks enter
NGSolve `ParallelFor` through `rad_hacapk_parallel.cpp`. Direct diagnostic `MatVec` calls are
TaskManager-preconditioned and should be made under a caller `TaskManager` scope.

## The validated picture (from the lab prototypes, all on `main`)

| Prototype (`examples/vim/`) | Result |
|---|---|
| `multipole_moment_hmatrix_compressibility.py` | Gate 1 PASS: the nonlocal moment kernel `D` (centroid field+grad coupling) has **bounded ACA rank** (field ~13, grad ~16-24), `N`-independent -> H-compressible. |
| `multipole_moment_matfree_solve.py` | Gate 2 PASS: matrix-free moment matvec reproduces the dense direct solve to `<1e-6` -> swapping the dense matvec for the HACApK matvec keeps the same answer. Element-block Jacobi is the best local preconditioner, but does NOT bound iters (grow `~dof^1.06`, mu_r-contrast driven). |
| `multipole_moment_scalable_path.py` | The A-build kernel is **cheap** (`~0.9 us`/(elem,face), single centroid->face integral) -- lighter than HDiv's face-face charge-Gram. Cheap local preconditioners did not produce bounded iterations in the prototype, which motivated trying H-LU; the later no-pivot HACApK diagnostic below showed that H-LU is not shippable for this non-symmetric moment matrix. |

Net: **SCALABLE multipole-moment MMM = HACApK A-build (cheap entries) + H-matvec BiCGSTAB + element-wise block Jacobi today.**  The preconditioner decision is not "any Jacobi":
scalar diagonal is rejected, identity fallback is disabled, and mixed-element models use each element's natural
`3/5/6` DOF block.  Bounded iterations remain future work (pivoted H-factor, redesigned global coarse space, or a
symmetrized moment formulation).

## The system

Per moment element, the rows are (`BuildMomentSystemCore`, `rad_interaction.cpp`):
3 dipole, 1 monopole, and the residual quadrupole rows needed to make the block
square.  Hex has 6 face-charge DOF (2 quadrupole rows); wedge/pyramid have 5 face-charge
DOF (1 residual quadrupole row). The dense system is

    A_raw = L  -  chi * R * C

- `L`  : the per-element LOCAL geometric-moment block (block-diagonal; 6x6 for hex,
         5x5 for wedge/pyramid; cheap).
- `C`  : the centroid field+grad coupling `C[e,k,g]` = field/grad component `k`
         (`k<3` = H, `3..8` = gradH) at element `e`'s centroid from unit charge on face
         DOF `g` (`BuildCentroidFieldGrad`; the cheap kernel, now IMA-aware).
- `R`  : the per-row linear combination (dipole = combo of `C[e,0:3]`, quad = combo of
         `C[e,3:9]`); sparse + local to element `e`.
- `chi`: susceptibility (per-element under Picard).

`A_raw`'s off-diagonal block for well-separated element clusters is `-chi*R*C`, a smooth
field/grad kernel folded by local moment functionals -> **low-rank** (Gate 1). So `A_raw`
is an H-matrix with the cluster tree over element centroids.

### KEY INSIGHT -- no row normalization needed for the H-LU path

`BuildMomentSystemCore` 2-norm-normalizes each row (for the dense GMRES/block-Jacobi
conditioning). **Row normalization is a diagonal row-scaling `D` and does NOT change the
exact solution of a direct solve:** `A_norm x = b_norm` with `A_norm=D A_raw`,
`b_norm=D b_raw`, is `A_raw x = b_raw` (multiply both sides by `D^{-1}`). So the H-LU
path builds the **un-normalized `A_raw`** and gets the SAME `x` as the normalized dense LU
(method 0) -- to solver tolerance. This removes the `O(N^2)` exact-row-norm precompute
(the row norm needs the dense field/grad part of every row); the H-matrix entry
`A_raw[i][j]` is then computable **on demand** from element geometry + the on-demand
`C[e_i, k, j]` (a single centroid->face evaluation).

## The C++ template to follow

`RadHACApKHDivSystemTet` (`rad_hacapk_hdiv.{h,cpp}`) already does exactly this shape for
the HDiv-VIM: it builds the system `A = M_mass + chi*N` as a HACApK H-matrix
(`RadHACApKBase` subclass with an on-demand `ComputeSystemEntry(i,j)`) and applies the
HACApK **H-LU** (`cHACApK_hlu_*`) as a scalable direct solve / strong preconditioner.
The moment manager mirrors it for the current scalable method-2 pure-hex path:

```
class RadHACApKMomentSystem : public RadHACApKBase {
  // ExtractCoordinates(): cluster-tree points = element centroids, expanded to the
  //   6 row-DOF (and 6 col-DOF) per hex co-located at the centroid (dof = 6*nHex).
  // ComputeSystemEntry(i,j) = A_raw[i][j]:
  //   decode i -> (element e_i, row-type in {dip_x,dip_y,dip_z,mono,quad0,quad1});
  //   decode j -> face DOF g (element e_j, local face);
  //   local part: if g in e_i's faces, add the L geometric-moment coefficient;
  //   nonlocal part: subtract chi * (row-type functional applied to C[e_i, :, g]),
  //     C[e_i,k,g] evaluated ON DEMAND (single centroid->face 8x8 Gauss, IMA mirrors).
  //   NO row normalization (see KEY INSIGHT).
  // SetSystemMode(chi) + cHACApK_hlu_* -> H-LU factor of A_raw -> scalable solve.
};
```

## Increments + verification gates

1. **On-demand entry.** A C++ `MomentSystemEntry(e_i, row_type, g)` (extract the
   single-(target,source) field/grad from `BuildCentroidFieldGrad`'s inner loop + the
   `BuildMomentSystemCore` row construction, un-normalized). **Gate:** entry-by-entry ==
   the dense `A_raw` from `BuildMomentSystemCore` (drop its row-norm) to machine precision.
2. **H-matrix build + matvec.** `RadHACApKMomentSystem` over element-centroid clusters.
   **Gate:** H-matvec `A_raw @ x` == dense `A_raw @ x` to ACA tolerance; ACA rank bounded
   in `N` (re-confirms Gate 1 in C++); build sub-cubic (Benchmark Policy JSON).
3. **~~H-LU solve~~ -> BLOCKED; use a Krylov solve on the H-matvec (REVISED 2026-06-22).**
   **FINDING (Increment 2.5 de-risk + the no-pivot diagnostic):** the HACApK H-LU
   (`cHACApK_hlu_*`) is **NO-PIVOT** (the HDiv template's `A = M_mass + chi*N` is SPD, so
   no-pivot is stable there).  The moment `A_raw` is **NON-symmetric** and, in the natural
   (element) ordering, the no-pivot factorization hits a **near-zero pivot**
   (`min|U_ii|/max|U_ii| ~ 2e-16`) even though the full matrix is well-conditioned
   (`cond(A_norm) ~ 1.6e3`) -- i.e. it is a PIVOTING (ordering) problem, NOT a scaling one.
   Measured: the no-pivot H-LU round trip is 8e-7 at dof=336 (all-dense, no truncation) but
   degrades to 6.7e-2 (1080) and DIVERGES to 1.4e+4 (2760) once ACA low-rank truncation
   compounds the near-zero pivot.  **Row/col equilibration does NOT fix it** (the near-zero
   pivot persists; a Python no-pivot LU on A_raw / A_norm / Ruiz-equilibrated all keep
   `min|U|/max|U| ~ 2e-16`).  dense LU (method 0) is fine because `dgesv` PIVOTS.  The
   prototype's "moment scales via H-LU" was an inference (cheap iterative preconditioners
   fail) that was never checked against the actual no-pivot H-LU -- it does not hold.
   **REVISED Increment 3:** solve method 2 with **GMRES/BiCGSTAB on the EXACT moment
   H-matvec (Increment 2, scalable storage) + a block-Jacobi preconditioner** (invert each
   element's local 6x6 `A_raw` block).  This avoids the factorization entirely; the
   matfree prototype (`multipole_moment_matfree_solve.py`) validated it converges (== dense to
   <1e-6) with iters that GROW ~dof^1.06 (the high-mu_r demag conditioning wall, shared with
   surface-charge MSC/HDiv-VIM -- a documented caveat, not a moment defect, NOT bounded like a true
   H-LU would give).  Route `radTRelaxationMethNo_2` (moment-eligible) to it; drop `Error204`.
   **Gate:** method-2 moment `x` == method-0 moment `x` to solver tol; storage scales (the
   dense matrix is gone); iter-growth documented.  Bounded-iter (a PIVOTED H-matrix factor,
   or a symmetrized moment formulation) stays FUTURE work.
4. **Nonlinear + storage decoupling. -> DONE (2026-06-22).** Two parts:
   - *Nonlinear (per-element chi):* `RadHACApKMomentSystem` gained a per-element-chi ctor; `SolveMomentHACApK`
     takes the `chiPerHex` vector (RHS `b[6h+t]=chi_h*Hext_h[t]`, per-element block-Jacobi), and the moment
     branch dropped its uniform-chi guard.  The Picard outer loop re-solves the H-system each iteration with
     the current chi -- the entry `MomentSystemEntry` already folds the row element's chi.  **Gate MET:**
     nonlinear C-yoke (MatSatIsoTab, driven to ~94-95% of Msat) -> method 2 == method 0 (external B ~1e-10,
     same Picard iteration count).  `tests/feec/test_multipole_moment_mmm.py::test_method2_nonlinear_matches_method0`,
     `examples/vim/verify_moment_nonlinear.py`.
   - *Storage decoupling (closes the Increment-3 storage gate that was a caveat):* the method-2 path no longer
     builds ANY dense O(N^2) buffer.  `SolveGen` sets `skipDenseMatrix=1` for all method 2 except B-input
     Newton/Hantila (no dense interaction N); `radTRelaxationMethNo_0::NeedsDenseMatrix()` returns false when
     `g_multipole_moment_hacapk` (no BaseMatrix); `SolveLinearStep` lazy-allocates the dgesv SystemMatrix (never
     reached on the H-BiCGSTAB happy path).  `Setup(skipDenseMatrix)` calls `PrecomputeHexaGeometry()` so the
     moment solve still sees the hexes (the index map normally built inside the skipped dense assembly).
     **Gate MET:** `bench_moment_storage_scaling.py` -- method2/method0 peak memory 0.69 -> 0.32 -> 0.18 ->
     0.12 across dof 1536..12288 (method 0 grows ~N^2, method 2 sub-quadratic ~N log N).
   - *Legacy callback acceleration port (2026_06_26):* before considering new optimizations, method 2 must
     first inherit the HACApK tricks from the previous 6-DoF implementation.  The current moment path now does: `skipDenseMatrix=1`;
     HACApK generation invalidation for thread-local callback caches; TaskManager-wrapped H-matrix build;
     a 6x6 single-entry + hash thread-local block cache in `RadHACApKMomentSystem`; and an `OnBeforeBuild`
     O(N) `PrecomputeMomentGeometry()` cache for face geometry, local moment rows, quadrupole test vectors, and
     Gauss samples.  Thus the callback no longer rebuilds row face geometry or source quadrature samples for
     every scalar entry; it computes one moment 6x6 block from cached geometry and reuses it across the 36 HACApK
     entry requests.  The geometry cache lookup is also hoisted out of the hot callback path: each worker thread
     grabs the shared O(N) geometry cache once, then uses a generation-checked thread-local pointer without a
     mutex.  The method-2 linear step now warm-starts BiCGSTAB from the previous Picard `sigma` and builds the
     block-Jacobi diagonal through the same cached 6x6 block path.  LAB sanity check: solid hex block, 1536 DOF,
     method 0 vs method 2 gives external-B relative difference `8.1e-9` and magnetization relative difference
     `6.5e-9`; method-2 H-matrix build changed from about `0.132 s` (block cache only) to `0.085-0.089 s` after
     the geometry precompute.  Nonlinear 144 DOF block still matches the method-0 parity test after warm start;
     a direct method-2 run used 116 outer Picard iterations and 957 total inner BiCGSTAB iterations.
   - *Legacy nonlinear H-matrix reuse port (2026_06_26):* the method-2 solve now builds a chi-free
     `K_geometry` H-matrix once per nonlinear solve and applies the current system as
     `A(chi)x = Lx + diag_row(chi) K_geometry x`, with `L` the exact block-diagonal local moment operator.
     Each Picard step updates only `chi`, the RHS, and the block-Jacobi local blocks.  This is the moment
     counterpart of the legacy "fixed geometry matrix + nonlinear diagonal/update" optimization and
     removes the previous Picard-times H-matrix rebuild.  LAB nonlinear 144 DOF block: wall time changed from
     about `0.949 s` to `0.045 s`; accumulated `t_hmatrix_build` changed from about `0.349 s` to `0.0039 s`,
     while the method-0 parity test still passes.
   - *Diagonal-block cache and allocation cleanup (2026_06_26):* the nonlinear context now also caches the
     per-element chi-independent local block `L_h` and the chi-free diagonal kernel block `K_{hh}`.  The
     block-Jacobi preconditioner for each Picard step is assembled as `L_h + chi_h K_{hh}` instead of calling
     the moment integral callback again for every diagonal block.  The BiCGSTAB matvec reuses its `Kx`
     workspace, the 6x6 LAPACK inversion path uses stack storage, small moment-local vector operations avoid
     TaskManager launch overhead, and the HACApK moment callback cache key includes the `kernelOnly` mode.
     LAB nonlinear 144 DOF block still uses 116 Picard iterations and about 961 total inner BiCGSTAB
     iterations; warmed same-process runs gave `t_linear_solve` about `0.012 s` and wall time about
     `0.016-0.017 s`, versus about `0.035 s` / `0.045 s` before this cleanup.
   - *Preconditioner audit before moving to nonlinear acceleration (2026_06_26):* within the class of local
     diagonal/block-diagonal preconditioners, the best production choice is the natural element block
     `A_hh(chi_h)=L_h+chi_h K_hh`.  The next stronger preconditioner should be global/hierarchical, not a
     casual local-block enlargement.  The archived
     `multipole_moment_block_size_test.py` result shows 2x2 and 3x3 local super-blocks worsen GMRES
     (174 -> 219/224 on the nxy=16 C-yoke), while only near-global blocks help.  A quick dense LAB check
     also showed that replacing the current `L_h + chi_h K_{hh}` diagonal by `L_h` alone gives no useful
     change on a small C-yoke (66 -> 67 iters).  The existing no-pivot HACApK H-LU remains unsuitable for
     direct insertion into the production moment path: previous probes show pivoting failure on non-symmetric
     `A_raw`, and any reusable H-LU preconditioner must be isolated from the matvec H-matrix because H-LU
     converts/factors the leaf tree in place.  Therefore the production path keeps element-block Jacobi here.
     A first 3-mode global dipole correction probe was tried and then removed: the two-stage smoke showed worse
     BiCGSTAB iterations in both engineering and high-mu stress cases.  The viable next research branches are
     (a) pivoted H-factorization, or (b) a redesigned global coarse space for demag/flux-path modes, starting
     from Stage A engineering cases.
   - *Element-block Jacobi is the local best choice (2026_06_26):* the minimum meaningful diagonal
     preconditioner unit, and therefore the best production local block-Jacobi unit, is one element block
     with the element's own DOF count.  For element `h`, let
     `q_h in R^{m_h}` be its local surface-charge/moment unknowns, where `m_h=3` for tetrahedral MMM,
     `m_h=5` for wedge/pyramid surface-charge moment elements, and `m_h=6` for hexahedral moment elements.
     The nonlinear Picard linear step has block form

         A_hh(chi_h) q_h + sum_{k != h} A_hk(chi_h) q_k = b_h,
         A_hh(chi_h) = L_h + chi_h K_hh.

     `L_h` is the Mathematica-derived local moment map from element charges to dipole/monopole/residual
     quadrupole rows; it is dense inside the element because the moment constraints intentionally mix the
     face DOF.  Therefore an element-block Jacobi preconditioner

         M_BD = blockdiag_h A_hh(chi_h)

     exactly removes the local moment problem and leaves only the nonlocal demagnetizing coupling:
     `M_BD^{-1} A = I + M_BD^{-1} A_off`.  This passes the zero-coupling consistency test: if all
     `A_hk, h != k` are removed, the preconditioned operator is exactly the identity.  A scalar diagonal
     Jacobi preconditioner `M_D = diag(A)` fails that same test because `M_D^{-1} A_hh` still contains the
     off-diagonal entries of `L_h + chi_h K_hh`; it does not even solve the isolated single-element moment
     constraints.  Mixed tet/wedge/hex systems therefore use mixed block sizes `3/5/6` by element, not a
     scalar diagonal and not an artificial uniform block.  Production code now fails loud if an element
     block inverse cannot be built; identity/scalar Jacobi fallback is disabled.
   - *Nonlinear initial-chi acceleration and no-silent-fallback policy (2026_06_26):* the Picard loop now
     initializes table-driven nonlinear materials from the local external-field magnitude when it is nonzero,
     instead of always starting from the near-zero-field ELF-style susceptibility.  This does not change the
     solver path or convergence criterion; it only starts Picard closer to the high-field operating point.
     LAB nonlinear 3x4x2 hex block (144 DOF, same BH table and drive as
     `test_method2_nonlinear_matches_method0`) changed from 116 to 60 outer iterations for both method 0
     and method 2; method-2 vs method-0 external-B norm differed by `4.9e-13` in the check run.  A trial
     moment Newton correction was not accepted because it failed to converge on this lock case; therefore
     `newton_method=True` now fails loud on the multipole-moment surface-charge path instead of silently
     running Picard.  Likewise method-2 HACApK build/solve failure no longer falls through to dense LU.
   - *HACApK parameter sweep and callback/batch audit (2026_06_26):* sweep JSON was collected in
     `C:\temp\radia_moment_hacapk_sweep_144dof_2026_06_26.json` and
     `C:\temp\radia_moment_hacapk_sweep_1536dof_2026_06_26.json`.  On the 1536-DOF nonlinear 8x8x4 hex
     block, all tested cases converged in 102 outer iterations.  In that run, `eps=1e-4, leaf=64, eta=2`
     was fastest (`wall ~1.41 s`, `t_hmatrix_build ~0.080 s`, `t_linear_solve ~1.32 s`, 1664 accumulated
     inner BiCGSTAB iterations); `eps=1e-5, leaf=16, eta=2` was close (`wall ~1.50 s`, 1852 inner
     iterations).  The HACApK C interface still invokes scalar `cHACApK_entry_ij` callbacks; the production
     moment path already computes cached 6x6 blocks behind that scalar API.  `RadHACApKMomentSystem` exposes
     `GetInteractionBlock6x6(elem_i, elem_j, block)` as the real block-level callback unit, and the scalar
     callback now routes through it.  A true vector/batched leaf fill still requires extending the HACApK C API,
     so no fake batch path is hidden here.
   - *Callback hot-path cleanup (2026_06_26):* `MomentSystemBlock6x6` now checks the thread-local geometry
     cache before touching the shared `g_momentGeomCache` mutex, so the HACApK scalar callback takes the
     mutex only on the first block per worker/generation.  The same function also fuses each source-face
     H/grad accumulation directly into the output 6x6 block column, avoiding the previous temporary
     `Hface[6][3]` / `Gface[6][6]` arrays and second face loop.  LAB sanity JSON:
     `C:\temp\radia_moment_callback_hotpath_bench_2026_06_26.json`.  In a single sequential run after this
     cleanup, the nonlinear 1536-DOF 8x8x4 hex block gave `t_hmatrix_build=0.062 s`,
     `t_linear_solve=1.38 s`, 1670 accumulated inner BiCGSTAB iterations, and 102 outer Picard iterations.
     The 144-DOF 3x4x2 lock case gave `t_hmatrix_build=0.0037 s`, 446 accumulated inner iterations, and
     60 outer Picard iterations.
   - *Residual caveat (unchanged from Increment 3):* iters still GROW with N (block-Jacobi only; the high-mu_r
     demag conditioning wall shared with surface-charge MSC / HDiv-VIM).  Bounded-iter (a pivoted H-factor or a
     symmetrized moment formulation) stays FUTURE work; it does not block Phase 3.

## Phase 3 (DONE, 2026-06-23): EIEM2 deleted -- multipole-moment MMM is canonical

With method 0/1/2 + IMA + nonlinear all on moment, the EIEM2 surface-charge collocation
kernel was REMOVED (live/dead refactor, commits bf4424d9/99556872/15f17022/d8d4ef99):
`radTInteraction::Compute6x6/5x5/MixedBlockFast`, the former `RadHACApKMSCManager` MSC machinery
(renamed `RadHACApKMMMManager` and now MMM-3x3-only), the dead IMA-mirror block, the per-face eval-point
caches, and the
`g_yano_eval_alpha`/`g_yano_no_center_charge`/`g_yano_pyramid_cloud` research flags (+ their
`SolverConfig` kwargs).  `MscEvalPoint` (alpha=0.5 midpoint) was ALSO deleted (final polish
`6b617a91`): the per-face MSC external-field branches (`dof>=5`/`dof==6`) in `SetupExternFieldArray`
/ `AddExternFieldFromMoreExtSource` are gone (only the tet `dof==3` fill remains), so `MscEvalPoint`
appears in NO source file.  The moment formulation samples the applied field at the element CENTROID
(the moment RHS via `BuildCentroidFieldGrad`), not per-face -- the per-face MSC fill was immaterial
to the converged moment result (it fed only the initial-H guess).  multipole-moment MMM (`BuildMomentSystemCore`
dense / `RadHACApKMomentSystem` H-matrix) is the SOLE surface-charge demag = the canonical
radia MMM for hex/wedge/pyramid soft iron (tet stays 3-DOF MMM).  The quadrupole rows are
the per-element residual eigenmodes (see `examples/vim/eigenmode_quadrupole_derivation.wls`).

## References

- `examples/vim/multipole_moment_{hmatrix_compressibility,matfree_solve,scalable_path}.py` (+ `.json`)
- `src/core/rad_hacapk_hdiv.{h,cpp}` (`RadHACApKHDivSystemTet`, the H-LU template)
- `src/core/rad_interaction.cpp` (`BuildCentroidFieldGrad`, `BuildMomentSystemCore`)
- `src/ext/HACApK_LH-Cimplm/` (`cHACApK_hlu_*` H-LU machinery)
